# 01b Data Cleaning - Standardized Input

## 1.1 Purpose

This notebook is a standardized-input version of the original data cleaning workflow. It is prepared for future company sales datasets that have already been mapped into the normalized schema by `scripts/standardize_raw_sales.py`.

This notebook is a preparation layer, not a replacement for `01_data_cleaning.ipynb` yet. It writes to `data/processed_standardized/` so the current validated project outputs in `data/processed/` are not overwritten.

## 1.2 Expected Input

The notebook consumes:

```text
data/interim/standardized_sales.csv
```

Required normalized columns:

- `invoice_no`
- `stock_code`
- `quantity`
- `invoice_date`
- `unit_price`

Optional normalized columns and defaults when absent:

- `description`: `"Description unavailable"`
- `customer_id`: missing values with pandas `string` dtype
- `country`: `"Unknown"`

`stock_code` is used as the normalized SKU key. `description` is treated only as a display field, so description variations do not split one stock code into multiple SKU profiles.

## 1.3 Cleaning Strategy

The cleaning process mirrors notebook 01 using the normalized column names:

1. Remove exact duplicate rows.
2. Separate cancellations and returns.
3. Remove invalid sales records with non-positive quantity or unit price.
4. Exclude records with unusable supplied descriptions; use the documented default when the optional description column is absent.
5. Retain missing customer IDs when the sales transaction is otherwise valid.
6. Exclude non-product transaction lines such as postage, fees, discounts, bank charges, and manual adjustments.
7. Create SKU-level and monthly SKU-level outputs for downstream analysis.

In [ ]:
import pandas as pd
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
standardized_path = project_root / "data" / "interim" / "standardized_sales.csv"
processed_dir = project_root / "data" / "processed_standardized"
processed_dir.mkdir(parents=True, exist_ok=True)

if not standardized_path.exists():
    raise FileNotFoundError(
        "Standardized input not found. Run scripts/standardize_raw_sales.py first. "
        f"Expected file: {standardized_path}"
    )

input_columns = pd.read_csv(standardized_path, nrows=0).columns
string_columns = {
    column: "string"
    for column in ["invoice_no", "stock_code", "description", "customer_id", "country"]
    if column in input_columns
}

df = pd.read_csv(
    standardized_path,
    dtype=string_columns,
)

print("Standardized input shape:", df.shape)
df.head()

## 1.4 Validate Standardized Schema

This step checks that the expected normalized fields are available before cleaning begins. The schema mapping and validation scripts should normally catch these issues earlier, but this check keeps the notebook self-explanatory and safe to run independently.

In [ ]:
required_columns = [
    "invoice_no",
    "stock_code",
    "quantity",
    "invoice_date",
    "unit_price",
]
optional_defaults = {
    "description": "Description unavailable",
    "customer_id": pd.NA,
    "country": "Unknown",
}
canonical_columns = [
    "invoice_no",
    "stock_code",
    "description",
    "quantity",
    "invoice_date",
    "unit_price",
    "customer_id",
    "country",
]

missing_required_columns = sorted(set(required_columns) - set(df.columns))
if missing_required_columns:
    raise ValueError(
        "Missing required standardized input columns: "
        + ", ".join(missing_required_columns)
    )

for column, default_value in optional_defaults.items():
    if column not in df.columns:
        if column == "customer_id":
            df[column] = pd.Series(pd.NA, index=df.index, dtype="string")
        else:
            df[column] = default_value

df = df[canonical_columns].copy()

df["invoice_no"] = df["invoice_no"].astype("string").str.strip()
df["stock_code"] = df["stock_code"].astype("string").str.strip()
df["description"] = df["description"].astype("string")
df["customer_id"] = df["customer_id"].astype("string").str.strip()
df["country"] = df["country"].astype("string").str.strip()
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")
df["invoice_date"] = pd.to_datetime(df["invoice_date"], errors="coerce")

print("Missing values:")
print(df.isna().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

## 1.5 Clean Valid Sales Transactions

The main sales dataset should represent valid physical product demand. Cancellations and returns are saved separately, while valid sales records are retained only when quantity, unit price, product code, product description, and invoice date are usable.

In [ ]:
df_clean = df.drop_duplicates().copy()

returns_cancellations = df_clean[
    df_clean["invoice_no"].str.startswith("C", na=False)
    | (df_clean["quantity"] < 0)
].copy()

sales = df_clean[
    (~df_clean["invoice_no"].str.startswith("C", na=False))
    & (df_clean["quantity"] > 0)
    & (df_clean["unit_price"] > 0)
    & (df_clean["stock_code"].notna())
    & (df_clean["description"].notna())
    & (df_clean["description"].str.strip() != "")
    & (df_clean["invoice_date"].notna())
].copy()

sales["description"] = sales["description"].str.strip().str.upper()
sales["stock_code"] = sales["stock_code"].str.strip()
sales["country"] = sales["country"].str.strip()

sales["invoice_date_only"] = sales["invoice_date"].dt.date
sales["invoice_month"] = sales["invoice_date"].dt.to_period("M").astype(str)
sales["invoice_week"] = sales["invoice_date"].dt.to_period("W").astype(str)
sales["revenue"] = sales["quantity"] * sales["unit_price"]

print("Rows after removing duplicates:", len(df_clean))
print("Valid sales rows before excluding non-product lines:", len(sales))
print("Returns / cancellations rows:", len(returns_cancellations))

sales.head()

## 1.6 Exclude Non-Product Transaction Lines

Some transaction lines are not physical product sales. These include postage, service fees, bank charges, discounts, marketplace fees, charity-related lines, and manual adjustments. These records are saved separately for transparency and excluded from the main product sales dataset.

In [ ]:
non_product_stock_codes = [
    "POST",        # Postage
    "DOT",         # Dotcom postage / service-related line
    "BANK CHARGES",
    "AMAZONFEE",
    "CRUK",
    "D",           # Discount
    "M",           # Manual adjustment
]

non_product_lines = sales[sales["stock_code"].isin(non_product_stock_codes)].copy()
sales = sales[~sales["stock_code"].isin(non_product_stock_codes)].copy()

print("Non-product transaction rows excluded:", len(non_product_lines))
print("Product sales rows after excluding non-product lines:", len(sales))

non_product_lines.head()

## 1.7 SKU Master Table

The SKU master table uses `stock_code` as the normalized SKU key. Product descriptions are retained as display fields and supporting context, not as the unique SKU identifier.

In [ ]:
sku_master = (
    sales
    .sort_values("invoice_date")
    .groupby("stock_code", as_index=False)
    .agg(
        description=("description", "first"),
        first_sale_date=("invoice_date", "min"),
        last_sale_date=("invoice_date", "max"),
        total_units=("quantity", "sum"),
        total_revenue=("revenue", "sum"),
        avg_unit_price=("unit_price", "mean"),
    )
)

print("SKU master shape:", sku_master.shape)
sku_master.head()

## 1.8 SKU Description Consistency Check

This table checks whether a single `stock_code` appears with multiple descriptions. The downstream SKU profile still uses `stock_code` as the key; description variation is treated as a data quality and reporting issue.

In [ ]:
sku_description_check = (
    sales
    .groupby("stock_code", as_index=False)
    .agg(
        description_count=("description", "nunique"),
        total_units=("quantity", "sum"),
        total_revenue=("revenue", "sum"),
    )
    .sort_values("description_count", ascending=False)
)

sku_description_check.to_csv(processed_dir / "sku_description_check.csv", index=False)
sku_description_check.head(20)

## 1.9 Monthly SKU-Level Sales Table

The cleaned transaction-level sales records are aggregated to `stock_code` and `invoice_month`. This creates a monthly SKU-level dataset that can support future classification, replenishment, and inventory analytics once the standardized pipeline is connected downstream.

In [ ]:
monthly_sku_sales = (
    sales
    .groupby(["stock_code", "invoice_month"], as_index=False)
    .agg(
        monthly_units=("quantity", "sum"),
        monthly_revenue=("revenue", "sum"),
        order_count=("invoice_no", "nunique"),
        avg_unit_price=("unit_price", "mean"),
    )
)

monthly_sku_sales = monthly_sku_sales.merge(
    sku_master[["stock_code", "description"]],
    on="stock_code",
    how="left",
)

monthly_sku_sales = monthly_sku_sales[
    [
        "stock_code",
        "description",
        "invoice_month",
        "monthly_units",
        "monthly_revenue",
        "order_count",
        "avg_unit_price",
    ]
]

monthly_sku_sales.to_csv(processed_dir / "monthly_sku_sales.csv", index=False)

print("Monthly SKU sales shape:", monthly_sku_sales.shape)
monthly_sku_sales.head()

## 1.10 Save Cleaned Outputs

All outputs are written to `data/processed_standardized/`. This keeps the standardized-input preparation workflow separate from the current validated outputs in `data/processed/`.

In [ ]:
sales.to_csv(processed_dir / "clean_sales.csv", index=False)
returns_cancellations.to_csv(processed_dir / "returns_cancellations.csv", index=False)
non_product_lines.to_csv(processed_dir / "non_product_lines.csv", index=False)
sku_master.to_csv(processed_dir / "sku_master.csv", index=False)

print("Saved files:")
print("- data/processed_standardized/clean_sales.csv")
print("- data/processed_standardized/returns_cancellations.csv")
print("- data/processed_standardized/non_product_lines.csv")
print("- data/processed_standardized/sku_master.csv")
print("- data/processed_standardized/sku_description_check.csv")
print("- data/processed_standardized/monthly_sku_sales.csv")

## 1.11 Data Quality Summary

The data quality summary documents the row-level transformation from standardized input to final cleaned product sales records.

In [ ]:
data_quality_summary = pd.DataFrame({
    "step": [
        "standardized_input",
        "after_duplicate_removal",
        "returns_cancellations_separated",
        "non_product_lines_excluded",
        "valid_product_sales_final",
        "monthly_sku_sales_records",
        "sku_master_records",
    ],
    "row_count": [
        len(df),
        len(df_clean),
        len(returns_cancellations),
        len(non_product_lines),
        len(sales),
        len(monthly_sku_sales),
        len(sku_master),
    ],
    "note": [
        "Original standardized transaction rows from data/interim/standardized_sales.csv",
        "Rows after removing exact duplicate records",
        "Cancellation and return-related records separated from main sales data",
        "Postage, fees, discounts, bank charges, manual adjustments, and other non-product lines excluded",
        "Final valid physical product sales records",
        "Monthly SKU-level aggregation records",
        "Normalized SKU master records using stock_code as SKU key",
    ],
})

data_quality_summary.to_csv(processed_dir / "data_quality_summary.csv", index=False)
data_quality_summary

## 1.12 Final Output Summary

This notebook creates a parallel set of standardized-input outputs:

1. `clean_sales.csv`
2. `returns_cancellations.csv`
3. `non_product_lines.csv`
4. `sku_master.csv`
5. `sku_description_check.csv`
6. `monthly_sku_sales.csv`
7. `data_quality_summary.csv`

These files are intentionally stored under `data/processed_standardized/` and do not replace the current project outputs.

In [ ]:
print("===== 01b Standardized-Input Data Cleaning Summary =====")
print("Valid product sales rows:", len(sales))
print("Returns / cancellations rows:", len(returns_cancellations))
print("Non-product transaction rows excluded:", len(non_product_lines))
print("Monthly SKU sales shape:", monthly_sku_sales.shape)
print("SKU master shape:", sku_master.shape)

print("\nOutput files:")
print("- data/processed_standardized/clean_sales.csv")
print("- data/processed_standardized/returns_cancellations.csv")
print("- data/processed_standardized/non_product_lines.csv")
print("- data/processed_standardized/sku_master.csv")
print("- data/processed_standardized/monthly_sku_sales.csv")
print("- data/processed_standardized/data_quality_summary.csv")
print("- data/processed_standardized/sku_description_check.csv")